In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
from pathlib import Path
import os
import pandas as pd
# Task 1: Write your code here:
delivery_path= os.path.join( path ,'Q1_data.csv')
df_delivery= pd.read_csv (delivery_path)

print(f"Dataset shape:{df_delivery.shape}")
df_delivery.head()

In [ ]:
# Task 2: Write your code here:
df_delivery.info()

In [ ]:
# Task 3: Write your code here:

df_delivery.describe()

In [ ]:
# Task 4: Write your code here:
import matplotlib.pyplot as plt



In [ ]:
# Price distribution (target variable)
plt.figure(figsize=(10, 5))
plt.hist(df_delivery['Distance_km'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery time')
plt.xlabel('distance')
plt.ylabel('Time')
plt.show()

In [ ]:
# Task 5: Write your code here:

In [ ]:
#  we will select and name the coloumns
cols = ['Order_ID','Distance_km', 'Weather','Traffic_Level', 'Vehicle_Type', 'Preparation_Time_min', 'Courier_Experience_yrs', 'Delivery_Time','Time_of_Day']
df_clean = df_delivery[cols].copy()

#drop order id coloumn
print(f"Before: {df_clean.shape}")
df_clean = df_clean.dropna(subset=['Order_ID'])
print(f"After dropping Order_ID : {df_clean.shape}")

In [ ]:

for col in ['Distance_km','Weather','Traffic_Level','Vehicle_Type','Preparation_Time_min','Delivery_Time']:
    df_clean[col] = df_clean[col].fillna('unknown')

df_clean['Courier_Experience_yrs'] = df_clean['Courier_Experience_yrs'].fillna(df_clean['Courier_Experience_yrs'].mode()[0])

print("Missing values remaining:", df_clean.isnull().sum().sum())

In [ ]:


from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder




In [ ]:
# Task 3:
categorical_cols = ['Distance_km', 'Weather', 'Traffic_Level', 'Vehicle_Type', 'Preparation_Time_min','Delivery_Time']
for col in categorical_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))

df_clean.head()

In [ ]:


from sklearn.model_selection import train_test_split, KFold


import warnings
warnings.filterwarnings('ignore')



In [ ]:

feature_cols = ['Distance_km', 'Weather', 'Traffic_Level', 'Vehicle_Type',
                'Preparation_Time_min', 'Delivery_Time']
X = df_clean[feature_cols]
y = df_clean['Distance_km']


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
print(f"\nFeature ranges - Min: {X_train.min().min():.2f}, Max: {X_train.max().max():.2f}")
X_train.head(3)

In [ ]:

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nScaled ranges - Min: {X_train_scaled.min():.2f}, Max: {X_train_scaled.max():.2f}")
pd.DataFrame(X_train_scaled, columns=X_train.columns).head(3)


In [ ]:
# Task 5: Write your code here:

In [ ]:
# Task 6: Write your code here:

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')



In [ ]:
# Task 1: Write your code here:

#i already did the step up there

In [ ]:
# Task 2,3,4,5: Write your code here:

In [ ]:
# Train Random Forest Regressor
model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
model.fit(X_train_scaled, y_train)
print("Model trained!")

In [ ]:
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []
rmse_scores = []

for train_idx, val_idx in kfold.split(X_train_scaled):
    X_fold_train, X_fold_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # Train and predict
    model.fit(X_fold_train, y_fold_train)
    y_fold_pred = model.predict(X_fold_val)

    # Calculate metrics
    mae_scores.append(mean_absolute_error(y_fold_val, y_fold_pred))
    rmse_scores.append(np.sqrt(mean_squared_error(y_fold_val, y_fold_pred)))

mae_scores = np.array(mae_scores)
rmse_scores = np.array(rmse_scores)

print(f"5-Fold CV Results:")
print(f"MAE:  ${mae_scores.mean():,.2f}")
print(f"RMSE: ${rmse_scores.mean():,.2f}")

In [ ]:
# Task 1: Write your code here:

In [ ]:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 5))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Predict and evaluate
y_pred = model.predict(X_test_scaled)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"MAE:  ${mae:,.2f}")
print(f"RMSE: ${rmse:,.2f}")

In [ ]:
# Task Bonus: Write your code here: